# 02 · From poker hands to features: a guided tour of Phase 2

Phase 1 found *what collusion looks like*. Phase 2 turns those observations into **numbers a model can learn from**, one row per pair of players. This notebook explains every design decision, shows the evidence that the pipeline is correct, and what the features already tell us before any model is trained.

Same structure as notebook 01: 🎯 Question · 🤔 Why it matters · 🔬 Method · 📊 What we saw · 💡 Inference · ➡️ Consequence.

### Sections
1. The pipeline at a glance
2. Is the plumbing right? Two independent correctness checks
3. The lift principle, proven on real data
4. Which features carry signal? The single-feature leaderboard
5. A hypothesis the data rejected: "stepping aside"
6. A warning for Phase 3: development vs evaluation shift
7. Summary and self-check

> Outputs use competition data, so they stay local. Run `python -m suspoker.pipeline --stage S5` first.

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

from suspoker.config import RAW_DIR
from suspoker.features import CATALOG
from suspoker.metric import TARGET_BEHAVIORS, average_precision
from suspoker.pipeline import FEATURES_DIR

pl.Config(tbl_rows=30, tbl_width_chars=180, fmt_str_lengths=50, float_precision=3)
pf = pl.read_parquet(FEATURES_DIR / "pair_features.parquet")
hf = pl.read_parquet(FEATURES_DIR / "hand_features.parquet")
def keyed(df):
    return df.with_columns(pl.min_horizontal("player_1", "player_2").alias("player_lo"),
                           pl.max_horizontal("player_1", "player_2").alias("player_hi"))
labels = keyed(pl.read_csv(RAW_DIR / "development_labels.csv"))
evals = keyed(pl.read_csv(RAW_DIR / "evaluation_pairs.csv"))
evidence = pl.read_csv(RAW_DIR / "development_evidence.csv")
dev = labels.join(pf.filter(pl.col("phase") == 0), on=["player_lo", "player_hi"])
print(f"pair-feature rows: {pf.height:,} (both periods) | features: {len([c for c in CATALOG if c in pf.columns])}")
print(f"hand-feature rows: {hf.height:,} | labeled development pairs with features: {dev.height:,}")

## 1 · The pipeline at a glance

```
raw parquet ─S1 ingest─▶ integer-coded tables (time-ordered by started_at)
            ─S2 strength─▶ hand rank of every postflop actor (phevaluator) + preflop percentile
            ─S3 interactions─▶ event tables:
                 • response events   (X answers Y's bet: fold / call / raise, ahead or behind?)
                 • pairwise flows    (chips of A that ended with B, in bb)
                 • heads-up streets  (only two players acted: checked through? strong hand present?)
                 • seat pairs        (every pair of seated players in every hand, incl. step-aside)
            ─S4 baselines─▶ each player's normal style per period (the yardstick)
            ─S5 features─▶ one row per (pair, period): 63 features in 7 families
                         + one row per (pair, candidate hand): 23 features for the evidence ranker
```

| Design choice | Why (which earlier finding) |
|---|---|
| Integer IDs, process 40 tables at a time | String-keyed operations were slow (DEC-012); pairs never cross tables (EDA §3), so chunking by table loses nothing |
| Hand order from `started_at` | Rule C1: only gameplay, never ID or file order |
| "Last aggressor" per street | EDA §7: who set the price is the core of partner-conditioned behavior |
| Strength-aware events (fold *while ahead*, pay off *while behind*) | EDA §8 walkthroughs: the decisive evidence needed hole cards |
| Pairwise flow = A's chips × B's share of the pot | Multiway pots don't split into pairs naturally; this does it exactly |
| Rates + shrunk **lift** | EDA §5: exposure is a decoy and differs by period; EDA §7: collusion is relative to the partner |
| Time blocks of 500 table-hands (F6) | Coordination is episodic. Blocks are cheaper than rolling windows over 30M pair-hand rows and capture the same idea |

**Runtime:** about 50 s for ingest and about 2.5 min for features, with a peak of about 4 GB RAM (budget: 45 min, 12 GB).

## 2 · Is the plumbing right? Two independent correctness checks

🎯 **Question:** How do we know the pairing, period split and hand filtering are correct, not just plausible?

🤔 **Why it matters:** A subtle bug here (an off-by-one in the period split, a missed seat) would silently corrupt every feature, and the model would still train happily.

🔬 **Method:** compare against numbers *the organizer computed independently*.
1. `evaluation_pairs.csv` lists `shared_hands` for every evaluation pair. Our pipeline computes the same quantity from scratch.
2. The 1,817 planted evidence hands must all survive our candidate filter for the evidence ranker.

In [ ]:
e = evals.join(pf.filter(pl.col("phase") == 1), on=["player_lo", "player_hi"], how="left")
print(f"evaluation pairs: {e.height:,} | missing: {e['n_shared'].null_count()} | "
      f"shared_hands mismatches: {e.filter(pl.col('n_shared') != pl.col('shared_hands')).height}")
lab_h = hf.filter(pl.col("phase") == 0).join(labels.select("pair_id", "player_lo", "player_hi"), on=["player_lo", "player_hi"])
kept = evidence.join(lab_h.select("pair_id", "hand_id"), on=["pair_id", "hand_id"], how="semi").height
print(f"evidence hands kept: {kept} / {evidence.height}")
per_pair = lab_h.group_by("pair_id").len()["len"]
print(f"candidate hands per labeled pair: median {per_pair.median():.0f} "
      f"(vs median ~110 shared development hands -> the filter removes ~60% of hands)")

📊 **What we saw:** **0 mismatches across 112,540 evaluation pairs**, and **1,817 / 1,817 evidence hands kept**, with the candidate set at about 40% of shared hands.

💡 **Inference:**
- Exact agreement on 112,540 independent counts is strong evidence that ingest, time ordering, the 60/40 period split and pair construction are all correct. A single misassigned hand per table would have produced thousands of mismatches.
- The candidate filter is **safe** (it loses no planted evidence) and **useful** (the evidence ranker gets about 40 candidates instead of about 110, so a random guess would already do better).
- The first version of the filter kept only 99.5% (9 missing). Investigating those 9 led to a **new feature** (section 5). **Checks are not just validation; they are a source of ideas.**

➡️ **Consequence:** the exposure-equality test is now a permanent unit test (`tests/test_features.py`).

## 3 · The lift principle, proven on real data

🎯 **Question:** Does comparing "behavior toward the partner" with "behavior toward everyone else" really beat raw behavior?

🤔 **Why it matters:** This is the core design bet of the whole project (SPEC §5.3). If raw rates worked just as well, all the "lift" machinery would be unnecessary complexity.

🔬 **Method:** For directed transfer, compare two versions of the same behavior (calling bets):
- **raw:** `call_rate_max`, the higher of the two players' overall call rate (against anyone)
- **lift:** `call_lift_max`, how much more often a player calls *the partner* than *others*, shrunk for sample size

**The formula, in words:** `base` = the player's call rate against everyone except the partner. `shrunk` = their call rate against the partner, pulled toward `base` by k = 20 pseudo-observations. `lift = shrunk − base`.

In [ ]:
y_dt = (dev["behavior_family"] == "directed_transfer").to_numpy().astype(int)
for f in ["call_rate_max", "call_lift_max"]:
    print(f"{f:<16} AP for directed transfer = {average_precision(y_dt, dev[f].to_numpy()):.3f}   (base rate {y_dt.mean():.3f})")

fig, ax = plt.subplots(figsize=(8, 5))
for lab, color, name in [(0, "#999999", "confirmed innocent"), (1, "#d62728", "directed transfer")]:
    m = (y_dt == lab) & ((dev["label"] == 0).to_numpy() | (y_dt == 1))
    ax.scatter(dev["call_rate_max"].to_numpy()[m], dev["call_lift_max"].to_numpy()[m], s=10, alpha=.6, c=color, label=name)
ax.axhline(0, lw=.8, c="k")
ax.set_xlabel("raw: overall call rate of the more 'cally' player"); ax.set_ylabel("lift: extra calling vs the partner")
ax.legend(); ax.set_title("Raw rate (x) doesn't separate colluders; lift (y) does"); plt.show()

cs = dev.filter(pl.col("call_rate_max") > dev["call_rate_max"].quantile(0.9))
print("pairs containing a 'calling station' (top 10% call rate):", cs.height)
print(cs.group_by("label").agg(pl.len().alias("pairs"), pl.col("call_lift_max").mean().alias("mean_lift")).sort("label"))

📊 **What we saw:** the raw call rate gives **AP 0.10** (barely above the 0.08 base rate). The lift version gives **AP 0.96**. In the scatter plot the red dots (colluders) sit *higher*, not further *right*. Calling-station pairs (innocent players who call a lot against everyone) have a lift near 0.04, and real colluders near 0.14.

💡 **Inference:**
- **Colluders aren't players who call a lot. They're players who call *their partner* a lot.** Measuring relative to each player's own habit removes the "player type" and leaves the "relationship".
- This is how the decoys in the competition (weak players, calling stations, tilt) are neutralized: their behavior is extreme but **uniform across opponents**, so their lift is about 0.
- **Shrinkage** keeps low-exposure pairs from producing extreme lifts by chance (see the unit test: 1/2 vs 20/40).

➡️ **Consequence:** confirms DEC-011. Every relationship feature is a lift. The raw style rates stay in the model only as *controls* (F7), so the model can learn interactions like "high lift *and* normal style".

## 4 · Which features carry signal? The single-feature leaderboard

🎯 **Question:** Before any model, how predictive is each feature *on its own*?

🔬 **Method:** for each feature, rank the 1,860 labeled development pairs by it (in whichever direction works better) and compute AP. Do this overall and per family (one family vs the rest). The base rates are 0.20 overall, 0.08 directed transfer, 0.07 soft play and 0.05 isolation.

In [ ]:
uni = pl.read_csv(FEATURES_DIR / "univariate.csv")
print(uni.select("feature", "family", "ap", "direction", *[f"ap_{f}" for f in TARGET_BEHAVIORS]).head(15))
print()
for fam in TARGET_BEHAVIORS:
    top = uni.sort(f"ap_{fam}", descending=True).head(3)
    print(f"{fam:<22}", "; ".join(f"{r['feature']} {r[f'ap_{fam}']:.2f}" for r in top.iter_rows(named=True)))
print()
print(uni.group_by("family").agg(pl.col("ap").max().alias("best_overall_ap")).sort("best_overall_ap", descending=True))

📊 **What we saw:**
| Family | Best single feature | AP (base rate) |
|---|---|---|
| Directed transfer | `call_lift_max`: calls the partner far more than others | **0.96** (0.08) |
| Soft play | `hu_strong_check_lift`: checks down *strong* hands heads-up with the partner | **0.82** (0.07) |
| Coordinated isolation | `squeeze_lift_max`: re-raises the partner in multiway pots | **0.74** (0.05) |
| Any collusion | `step_aside_lift_min` (negative direction, section 5) | **0.82** (0.20) |

F7 *controls* are weak on their own (best ≈ 0.30), which is intended. They describe the player, not the relationship.

💡 **Inference:**
- **Each family's best feature is exactly the one the EDA predicted** (EDA §7: calling the partner, never attacking the partner, re-raising behind the partner). The features measure what we meant them to measure.
- These are *single* features. A model combining them should do better, especially on soft play and isolation, where the best single feature is weaker.
- `sd_together_rate` scores well overall (0.65) despite being labeled "exposure". It's really behavioral: partners reach showdown together more often, which matches the soft-play check-down pattern.
- **Caution:** univariate AP on 1,860 pairs is an optimistic, in-sample number. It ranks features; it doesn't predict the model's score. The honest number comes from grouped cross-validation in P3.

➡️ **Consequence:** feature design is validated before modeling. In P3, families F3, F4 and F5 should drive the per-family behavior model (M2).

## 5 · A hypothesis the data rejected: "stepping aside"

🎯 **Question:** The 9 evidence hands the first filter missed were all *isolation* hands where one partner raised and the other **folded out of the way**. Do isolation partners systematically step aside for each other?

🔬 **Method:** new feature `step_aside_lift`: how much more often a player folds preflop **after the partner has entered the pot** than after anyone else has entered (shrunk lift, both directions).

In [ ]:
cols = ["ap", "direction", *[f"ap_{f}" for f in TARGET_BEHAVIORS]]
print(uni.filter(pl.col("feature").str.starts_with("step_aside")).select("feature", *cols))
print(dev.group_by("behavior_family").agg(pl.col("step_aside_lift_min").mean().round(3),
                                          pl.col("step_aside_lift_max").mean().round(3)).sort("behavior_family"))

📊 **What we saw:** the feature is strong overall (AP 0.82), but **in the negative direction**. Colluders fold *less* once their partner is in the pot. For isolation specifically it carries almost no signal (AP 0.07 vs base rate 0.05).

💡 **Inference:**
- The hypothesis ("isolation partners fold for each other") **is not supported** at pair level. Those 9 hands are real but rare. They are evidence hands, not a habit.
- The *opposite* behavior is a strong habit, mainly of **directed transfer and soft play**: they need to be in the same pot to transfer chips or to check it down. EDA §6 showed exactly this (both partners put chips in 99–100% of the time in their evidence hands). The feature measures it from the other side: "when my partner enters, I don't fold."
- **Lesson:** when a feature works for a different reason than expected, believe the data, but explain it. We now understand *why* it works, so it's not a mystery feature we'd be afraid to trust.

➡️ **Consequence:** keep `step_aside_lift` (pair level) and `h_step_aside` (hand level, which fixed the evidence coverage to 100%). Record the rejected hypothesis honestly.

## 6 · A warning for Phase 3: development vs evaluation shift

🎯 **Question:** Do features look the same in the development period (where we train) and the evaluation period (where we're scored)?

🤔 **Why it matters:** A model learns thresholds like "lift above 0.05 means collusion". If evaluation features are systematically smaller, the same pair would score lower, and thresholds learned in development would be miscalibrated.

In [ ]:
rows = []
for f in ["n_shared", "call_lift_max", "step_aside_lift_min", "hu_strong_check_lift", "squeeze_lift_max",
          "netflow_bb100", "blk_call_lift_max"]:
    s = pf.group_by("phase").agg(pl.col(f).median().alias("median"), pl.col(f).std().alias("std")).sort("phase")
    rows.append({"feature": f, "dev_median": s["median"][0], "eval_median": s["median"][1],
                 "dev_std": s["std"][0], "eval_std": s["std"][1]})
pl.DataFrame(rows)

📊 **What we saw:** evaluation pairs share fewer hands (median 62 vs 95). Lift features move slightly **toward 0** in evaluation (for example `squeeze_lift_max` 0.008 → 0.000, `blk_call_lift_max` 0.039 → 0.030). Flow rates get **noisier** (the std of `netflow_bb100` goes 80 → 121).

💡 **Inference:** this is the expected side effect of shrinkage. **Fewer hands means stronger pull toward the baseline, so smaller lifts.** Block maxima also shrink because evaluation has 4 time blocks instead of 6, and the maximum of fewer numbers is smaller. Since AP depends only on **ranking within the evaluation set**, a uniform shrink doesn't hurt. What could hurt is *uneven* shift between features, because the model would rebalance them wrongly.

➡️ **Consequence for P3:** compare OOF and Kaggle scores carefully. If the gap is large, train on development features recomputed on time windows matching evaluation exposure (the SPEC §11 mitigation). Block-max features may need normalizing by the number of blocks.

## 7 · Summary and self-check

| # | Finding | Why it matters |
|---|---|---|
| 1 | Shared-hand counts match the organizer exactly (112,540 / 112,540) | The data plumbing is correct |
| 2 | Candidate filter keeps 100% of evidence while removing ~60% of hands | The evidence ranker has an easier, safe search space |
| 3 | Raw call rate AP 0.10 vs lift AP 0.96 for directed transfer | The lift principle works; decoys are neutralized |
| 4 | Each family's best feature is the one the EDA predicted | Features measure what we intended |
| 5 | Step-aside works, but backwards (colluders *join* the partner's pots) | A rejected hypothesis became an understood signal |
| 6 | Evaluation lifts are slightly smaller and noisier | Watch the OOF-vs-Kaggle gap in P3 |

### Self-check
1. Why does a calling station get a lift near 0 even though they call constantly?
2. Why is matching the organizer's `shared_hands` such strong evidence, compared with "the numbers look reasonable"?
3. A pair has 2 confrontations, and one player folded both times to the partner (100%) against a 40% habit. Roughly what lift do they get with k = 20, and why is that desirable?
4. Why can lifts shrink in the evaluation period without hurting AP, and when *would* it hurt?

<details><summary>Answers</summary>

1. Lift compares the player with themselves: calls vs the partner minus calls vs everyone else. A calling station calls everyone, so the difference is about 0.
2. It's an exact, independent check on 112,540 numbers. Any systematic bug (period boundary, seats, pairing) would produce many mismatches, while "reasonable-looking" numbers can hide them.
3. shrunk = (2 + 20 × 0.4) / (2 + 20) ≈ 0.45, so the lift is about +0.05, small. Two hands are weak evidence, so the feature stays near "normal" until more data accumulates.
4. AP only uses the ranking *within* the evaluation set, so a uniform shrink keeps the order. It hurts if different features shrink by different amounts, because the model then weighs them incorrectly.
</details>